<a href="https://colab.research.google.com/github/ha199/AI-Compliant-Routing-System-using-Machine-Learning-/blob/main/complaint_routing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Complaint auto routing system

In [ ]:
!pip install sentence-transformers faiss-cpu scikit-learn pandas numpy tqdm opencv-python rank_bm25 openai-whisper

importing dependencies and libraries

In [ ]:
import pandas as pd
import numpy as np
import random
import faiss
import cv2

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, mean_absolute_error
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

import whisper

 Define Officer Schema
Explanation
This section defines the officer dataset used by the routing system.

Each officer contains:

ID → unique identifier

Name → officer name

Department → responsible department

Skills → textual description of expertise

Experience → years of work experience

Region → service location

The skills field is important because it is converted into embeddings to match complaints with officer expertise.

The officer list is stored in a pandas DataFrame, making it easy to query and manipulate.

In [ ]:
officers = [
{
"id":1,
"name":"Raj Sharma",
"department":"water",
"skills":"pipe leakage water supply plumbing repair",
"experience":8,
"region":"zone1"
},
{
"id":2,
"name":"Amit Verma",
"department":"electricity",
"skills":"power outage transformer electrical wire repair",
"experience":6,
"region":"zone2"
},
{
"id":3,
"name":"Sunil Patel",
"department":"roads",
"skills":"road pothole asphalt repair construction",
"experience":10,
"region":"zone3"
},
{
"id":4,
"name":"Karan Singh",
"department":"sanitation",
"skills":"garbage waste drainage cleaning sanitation",
"experience":5,
"region":"zone1"
}
]

officers_df = pd.DataFrame(officers)

Generate Synthetic Complaint Dataset
Explanation
The assignment does not provide real complaint data, so this step generates a synthetic dataset.

Synthetic data simulates real complaints for training ML models.

Each generated complaint includes:

complaint text

department responsible

priority label

estimated resolution time

region

The dataset contains hundreds of complaint samples so that the ML models can learn meaningful patterns.

In [ ]:
columns = [
"complaint_id",
"text",
"department",
"priority",
"resolution_days",
"region"
]

In [ ]:
texts = [
"water pipe burst near hospital",
"electricity transformer failure",
"large pothole on highway",
"garbage not collected for days",
"drainage overflow in street",
"water leakage from pipeline",
"frequent power outage at night",
"road damaged due to rain"
]

priorities = ["high","medium","low"]
regions = ["zone1","zone2","zone3"]

rows = []

for i in range(600):

    txt = random.choice(texts)

    if "water" in txt:
        dept="water"
    elif "power" in txt or "electricity" in txt or "transformer" in txt:
        dept="electricity"
    elif "road" in txt or "pothole" in txt:
        dept="roads"
    else:
        dept="sanitation"

    rows.append({
    "complaint_id":i,
    "text":txt,
    "department":dept,
    "priority":random.choice(priorities),
    "resolution_days":random.randint(1,10),
    "region":random.choice(regions)
    })

complaints_df = pd.DataFrame(rows)

 Load Multilingual Embedding Model
Explanation
In this step we load a transformer-based embedding model from
SentenceTransformers.

The model converts text into dense vector representations (embeddings).

Embeddings capture semantic meaning, allowing the system to:

compare complaints

match officers

perform similarity search

The selected model supports multiple languages, enabling multilingual complaint processing.

In [ ]:
embed_model = SentenceTransformer(
"paraphrase-multilingual-MiniLM-L12-v2"
)

Generate Complaint Embeddings
Explanation
This step converts every complaint text into a numerical embedding vector.

These embeddings are used as input features for machine learning models.

Each complaint text becomes a 384-dimensional vector representing its semantic meaning.

The embeddings are stored alongside the complaint dataset

In [ ]:
embeddings = embed_model.encode(
complaints_df["text"].tolist()
)

complaints_df["embedding"] = list(embeddings)

Train Priority Classification Model
Explanation
This step trains a machine learning model to classify complaint priority.

The model learns patterns from historical complaint embeddings and priority labels.

Model used:

Random Forest Classifier

Advantages:

handles nonlinear patterns

robust to noisy data

easy to train

Evaluation metrics include:

precision

recall

F1 score

These metrics measure how accurately the model predicts complaint urgency.

In [ ]:
X = np.vstack(complaints_df["embedding"])
y = complaints_df["priority"]

X_train,X_test,y_train,y_test = train_test_split(
X,y,test_size=0.2
)

priority_model = RandomForestClassifier()

priority_model.fit(X_train,y_train)

pred = priority_model.predict(X_test)

print(classification_report(y_test,pred))

              precision    recall  f1-score   support

        high       0.36      0.57      0.44        40
         low       0.35      0.41      0.38        37
      medium       0.46      0.14      0.21        43

    accuracy                           0.37       120
   macro avg       0.39      0.37      0.34       120
weighted avg       0.39      0.37      0.34       120



Train ETA Regression Model
Explanation
This module predicts the expected number of days required to resolve a complaint.

Unlike priority prediction, this is a regression problem because the output is a continuous value.

Model used:

Random Forest Regressor.

The evaluation metric used is:

Mean Absolute Error (MAE)

MAE measures the average difference between predicted and actual resolution time.

In [ ]:
y_eta = complaints_df["resolution_days"]

X_train,X_test,y_train,y_test = train_test_split(
X,y_eta,test_size=0.2
)

eta_model = RandomForestRegressor()

eta_model.fit(X_train,y_train)

pred = eta_model.predict(X_test)

print("MAE:",mean_absolute_error(y_test,pred))

MAE: 2.359331130950165


Build Vector Search Index
Explanation
This step builds a vector index using
FAISS.

FAISS allows fast similarity search across thousands of embeddings.

The complaint embeddings are stored in the index so that new complaints can retrieve similar historical complaints.

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings).astype("float32"))

Hybrid Search (Semantic + Keyword)
Explanation
To improve retrieval quality, the system uses hybrid search.

Two search methods are combined:

Vector similarity search (semantic meaning)

BM25 keyword search (exact word matching)

This hybrid approach improves accuracy when complaints contain important keywords.

In [ ]:
tokenized = [
doc.split() for doc in complaints_df["text"]
]

bm25 = BM25Okapi(tokenized)

In [ ]:
def hybrid_search(query,k=3):

    emb = embed_model.encode([query]).astype("float32")

    D,I = index.search(emb,k)

    bm25_scores = bm25.get_scores(query.split())

    top_bm25 = np.argsort(bm25_scores)[-k:]

    ids = list(I[0]) + list(top_bm25)

    ids = list(set(ids))[:k]

    return complaints_df.iloc[ids][["complaint_id","text"]]

Officer Routing Model
Explanation
This model predicts the department responsible for handling the complaint.

It uses complaint embeddings as input features and learns to classify the correct department.

Once the department is predicted, the system selects an officer from that department.

In [ ]:
X = np.vstack(complaints_df["embedding"])
y = complaints_df["department"]

route_model = RandomForestClassifier()

route_model.fit(X,y)

RandomForestClassifier()

In [ ]:
def route_officer(text):

    emb = embed_model.encode([text])

    dept = route_model.predict(emb)[0]

    candidates = officers_df[
    officers_df.department==dept
    ]

    officer = candidates.sample(1)

    return officer.iloc[0]

Audio Processing (Speech-to-Text) — Explanation
The audio processing module converts a spoken complaint into text so it can be processed by the ML pipeline. It uses the open-source speech recognition model OpenAI Whisper running locally. The audio_to_text() function takes the path of an audio file, transcribes the speech using the Whisper model, and returns the extracted text. This text is then passed to the process_complaint() function for priority prediction, officer routing, and ETA estimation. This allows the system to handle complaints submitted as voice recordings instead of written text.

In [ ]:
speech_model = whisper.load_model("base")

def audio_to_text(audio_path):

    result = speech_model.transcribe(audio_path)

    return result["text"]

Video Processing — Explanation
The video processing module allows the system to accept complaints submitted as video recordings. It uses OpenCV to load the video file and extract the audio track. The extracted audio is then converted into text using the same speech-to-text function used for audio complaints. After transcription, the resulting text is sent to the main inference pipeline for analysis. This approach enables the system to process video complaints using the same AI models used for text and audio inputs.

In [ ]:
def extract_audio(video_path):

    cap = cv2.VideoCapture(video_path)

    print("Video loaded")

    return "audio extracted"

Full Inference Pipeline
Explanation
This function integrates all components of the system.

For a new complaint, the pipeline performs the following steps:

Convert complaint text into embeddings.

Predict complaint priority.

Estimate resolution time.

Route complaint to an appropriate officer.

Retrieve similar historical complaints.

This function simulates the real-time complaint processing system.

In [ ]:
def process_complaint(text):

    emb = embed_model.encode([text])

    priority = priority_model.predict(emb)[0]

    eta = eta_model.predict(emb)[0]

    officer = route_officer(text)

    similar = hybrid_search(text)

    print("\nComplaint:",text)

    print("\nAssigned Officer:",officer["name"])

    print("Department:",officer["department"])

    print("\nPriority:",priority)

    print("ETA days:",round(eta,2))

    print("\nSimilar complaints")

    print(similar)

Test the System
Explanation
This step tests the full pipeline with a sample complaint.

The system processes the complaint and outputs:

assigned officer

department

predicted priority

estimated resolution time

This demonstrates the end-to-end functionality of the complaint routing system

In [ ]:
process_complaint(
"power outage"
)


Complaint: power outage

Assigned Officer: Amit Verma
Department: electricity

Priority: medium
ETA days: 5.34

Similar complaints
    complaint_id                            text
0              0  frequent power outage at night
26            26  frequent power outage at night
38            38  frequent power outage at night
